# RFx Co-pilot (Guided Builder)

This notebook demonstrates an active LLM agent that guides the user to build an RFx step-by-step.
1. **Scope**: What are we buying?
2. **Line Items**: Searching the catalog (IT, Office Supplies, Packaging) and adding SKUs/Generic Requirements.
3. **Questionnaire**: Adding custom questions for the vendors to answer (e.g., SLAs, Warranty, ISO certs).
4. **View**: Shows the fully assembled RFx.

In [ ]:
import sys
import os
import json
import pandas as pd
import urllib.request
import urllib.parse
from IPython.display import display, HTML

# Add root path so we can use existing app.llm client connection
sys.path.append(os.path.abspath('.'))
from app import llm, config

# 1. Load the Item Master data (Make sure you ran generate_mock_data.py to add IT & Office supplies!)
df = pd.read_csv('../Data/aerchain_dataset/01_master_data/item_master.csv')

# 2. Define our agent tools
def search_catalog(keywords=None, category=None):
    """Searches the catalog and returns matching SKUs across all categories."""
    results = df.copy()
    
    if category:
        results = results[results['category'].str.contains(category, case=False, na=False)]
        
    if keywords:
        for kw in keywords:
            # Strip trailing 's' for basic singular matching (e.g., 'laptops' -> 'laptop')
            search_kw = kw[:-1] if kw.lower().endswith('s') and len(kw) > 3 else kw
            
            # Search across description, category, and sub_category
            mask = (
                results['description'].str.contains(search_kw, case=False, na=False) |
                results['category'].str.contains(search_kw, case=False, na=False) |
                results['sub_category'].str.contains(search_kw, case=False, na=False)
            )
            results = results[mask]
            if results.empty:
                return []
                
    # Return top 15 matches to keep the LLM context clean
    return results[['item_code', 'description', 'category', 'sub_category', 'status']].head(50).to_dict('records')

def search_internet(query):
    """Searches Wikipedia for hardware specs or general knowledge."""
    url = f"https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch={urllib.parse.quote(query)}&utf8=&format=json"
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req) as response:
            data = json.loads(response.read())
            results = data.get('query', {}).get('search', [])
            return [{"title": r["title"], "snippet": r["snippet"]} for r in results[:3]]
    except Exception as e:
        return f"Error searching Wikipedia: {e}"


tools = [
    {
        "type": "function",
        "function": {
            "name": "search_catalog",
            "description": "Search the item master catalog. IMPORTANT: ONLY pass the primary noun (e.g. 'MacBook Pro', 'Laptop', 'Mouse') in keywords. Do NOT pass specific specs like '32GB' or 'GPU' into the search tool; instead, search for the broad category and filter the results yourself.",
            "parameters": {
                "type": "object",
                "properties": {
                    "keywords": {"type": "array", "items": {"type": "string"}, "description": "Keywords to search"},
                    "category": {"type": "string", "description": "Optional broad category filter"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "set_rfx_scope",
            "description": "Set the overarching project scope or title for the RFx (e.g., 'IT Hardware Refresh 2026').",
            "parameters": {
                "type": "object",
                "properties": {
                    "scope": {"type": "string", "description": "The RFx project title or purpose"}
                },
                "required": ["scope"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_to_rfx",
            "description": "Add specific item codes OR a generic requirement to the RFx draft, including site quantities.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item_codes": {
                        "type": "array", 
                        "items": {"type": "string"},
                        "description": "Specific item codes (e.g. IT-LAP-0001) to add."
                    },
                    "generic_requirements": {
                        "type": "string",
                        "description": "Description of minimum/maximum acceptable limits if the user doesn't pick an exact SKU (e.g., 'Any 14-inch laptop with i7 and 16GB RAM')."
                    },
                    "site_quantities": {
                        "type": "object",
                        "additionalProperties": {"type": "integer"},
                        "description": "Quantities needed per site (e.g., {'Hosur': 100, 'Bangalore': 50})."
                    }
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "set_questionnaire",
            "description": "Sets or replaces the entire list of questions for suppliers to answer. This COMPLETELY OVERWRITES the previous list, so always provide the FULL list of desired questions.",
            "parameters": {
                "type": "object",
                "properties": {
                    "questions": {
                        "type": "array", 
                        "items": {"type": "string"},
                        "description": "Full list of questions to ask suppliers."
                    }
                },
                "required": ["questions"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "remove_from_rfx",
            "description": "Remove one or more specific line items from the RFx draft by their 1-based line numbers.",
            "parameters": {
                "type": "object",
                "properties": {
                    "line_numbers": {"type": "array", "items": {"type": "integer"}, "description": "The 1-based line item numbers to remove (e.g. [1, 2, 5])."}
                },
                "required": ["line_numbers"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_internet",
            "description": "Search the internet (Wikipedia) to answer domain-specific or hardware questions (e.g. \"Does M3 Max have a GPU?\").",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query."}
                },
                "required": ["query"]
            }
        }
    }
]

In [ ]:
class GuidedCopilot:
    def __init__(self):
        self.rfx_state = {
            "scope": None,
            "items": [],
            "questionnaire": []
        }
        self.client = llm.client()
        
        system_prompt = """You are 'Pro', a helpful, efficient procurement copilot at AerChain. Your persona is sharp, friendly, and concise. You ALWAYS respond in very short messages (1-3 sentences max).
Your job is to actively guide the user to build a complete RFx step-by-step. Do not let them just search and leave. Guide them through these steps, one at a time:
1. Scope: Ask what the overarching goal of the procurement is. Use set_rfx_scope to save it.
2. Line Items: Help them search the catalog. Use add_to_rfx to save items.
   *DRILL DOWN GUIDANCE*: If the user searches for a broad category and you find many results, DO NOT just list SKUs. Summarize the broad *types* available (e.g. "We have Ergonomic and Gaming mice. Which do you prefer?") to help them drill down.
   *CLARIFYING SPECS*: If a user's request for complex IT hardware (like a laptop) is underspecified, DO NOT immediately call add_to_rfx. Ask follow-up questions to clarify the exact specs (e.g., "Do you need M2 or M3 chips?", "What screen size?") BEFORE saving it.
   *NO QUANTITIES IN DESCRIPTIONS*: NEVER include the numerical quantity in the `generic_requirements` field. The quantity belongs ONLY in the `site_quantities` argument. The generic requirement should only describe the item (e.g., "MacBook Pro, 32GB RAM", NOT "10 MacBook Pros").
   *INTERNET SEARCH*: If the user asks a technical or domain-specific question (e.g., "Does the M3 Max have a GPU?"), use the search_internet tool to gather factual information before answering.
3. Questionnaire: Ask them if they want to include a questionnaire for vendors. Use set_questionnaire to save their questions. Note that set_questionnaire COMPLETELY OVERWRITES the previous list, so if the user asks you to modify a question, rewrite the entire list and pass it in.
Important rules:
- Remind them of what is remaining if they miss a step.
- Finish ONE STEP at a time.
- Keep your answers very short and conversational.
"""
        
        self.messages = [{"role": "system", "content": system_prompt}]

    def chat(self, user_text):
        self.messages.append({"role": "user", "content": user_text})
        
        while True:
            response = self.client.chat.completions.create(
                model=config.MODEL,
                messages=self.messages,
                tools=tools,
                reasoning_effort="none"
            )
            
            message = response.choices[0].message
            self.messages.append(message)
            
            if message.content:
                print(f"\n\033[92mPro:\033[0m {message.content}")
                
            if not message.tool_calls:
                break
                
            for tool_call in message.tool_calls:
                print(f"\033[90m  [System: Executing {tool_call.function.name}(...)]\033[0m")
                args = json.loads(tool_call.function.arguments)
                
                if tool_call.function.name == "search_catalog":
                    result = search_catalog(**args)
                    self.messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)})
                    print(f"\033[90m  [System: Found {len(result)} items]\033[0m")
                    
                elif tool_call.function.name == "set_rfx_scope":
                    self.rfx_state["scope"] = args.get("scope")
                    self.messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": "Scope saved successfully."})
                    
                elif tool_call.function.name == "add_to_rfx":
                    start_len = len(self.rfx_state["items"])
                    site_quantities = args.get('site_quantities', {})
                    if not site_quantities:
                        self.rfx_state["items"].append({
                            "item_codes": args.get('item_codes', []),
                            "generic_requirements": args.get('generic_requirements', ""),
                            "location": "N/A",
                            "quantity": "N/A"
                        })
                    else:
                        for site, qty in site_quantities.items():
                            self.rfx_state["items"].append({
                                "item_codes": args.get('item_codes', []),
                                "generic_requirements": args.get('generic_requirements', ""),
                                "location": site,
                                "quantity": qty
                            })
                    end_len = len(self.rfx_state["items"])
                    added_nums = list(range(start_len + 1, end_len + 1))
                    self.messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": f"Items added successfully at Line Numbers: {added_nums}"})
                    
                elif tool_call.function.name == "remove_from_rfx":
                    line_nums = args.get("line_numbers", [])
                    if not isinstance(line_nums, list):
                        line_nums = [line_nums]
                    line_nums.sort(reverse=True)
                    removed = []
                    errors = []
                    for num in line_nums:
                        idx = num - 1
                        if 0 <= idx < len(self.rfx_state["items"]):
                            self.rfx_state["items"].pop(idx)
                            removed.append(num)
                        else:
                            errors.append(num)
                    msg = f"Removed line items: {removed}."
                    if errors: msg += f" Errors (not found): {errors}."
                    self.messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": msg})
                    
                elif tool_call.function.name == "search_internet":
                    query = args.get("query", "")
                    result = search_internet(query)
                    self.messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": json.dumps(result)})
                    print(f"\033[90m  [System: Searched internet for '{query}']\033[0m")
                    
                elif tool_call.function.name == "set_questionnaire":
                    self.rfx_state["questionnaire"] = args.get("questions", [])
                    self.messages.append({"role": "tool", "tool_call_id": tool_call.id, "content": "Questionnaire completely replaced/set successfully."})
                    
                    
    def view_rfx(self):
        display(HTML("<h2>RFx Draft Summary</h2>"))
        
        # 1. Scope
        scope = self.rfx_state["scope"] or "<i>Not Defined Yet</i>"
        display(HTML(f"<b>Project Scope:</b> {scope}<br><br>"))
        
        # 2. Line Items
        if self.rfx_state["items"]:
            rows = []
            for idx, item in enumerate(self.rfx_state["items"]):
                descriptions = []
                
                if item.get("item_codes"):
                    descs = df[df['item_code'].isin(item["item_codes"])]['description'].tolist()
                    descriptions.extend(descs)
                    
                if item.get("generic_requirements"):
                    descriptions.append(item["generic_requirements"])
                    
                item_desc_str = " | ".join(descriptions) if descriptions else "N/A"
                
                rows.append({
                    "Line Item": idx + 1,
                    "Item Description": item_desc_str,
                    "Location": item.get("location", "N/A"),
                    "Quantity": item.get("quantity", "N/A")
                })
                
            display(HTML("<b>Line Items:</b>"))
            display(pd.DataFrame(rows).set_index("Line Item"))
        else:
            display(HTML("<b>Line Items:</b> <i>None</i><br>"))
            
        # 3. Questionnaire
        display(HTML("<br><b>Supplier Questionnaire:</b>"))
        if self.rfx_state["questionnaire"]:
            q_list = "<ul>" + "".join(f"<li>{q}</li>" for q in self.rfx_state["questionnaire"]) + "</ul>"
            display(HTML(q_list))
        else:
            display(HTML("<i> None</i>"))

agent = GuidedCopilot()


In [ ]:
print("\033[92mPro:\033[0m Hey there! I'm Pro. Let's build your RFx together.")
print("     (Type 'END' to exit, or 'VIEW' to see your RFx draft)\n")

while True:
    user_input = input("You: ")
    if not user_input.strip():
        continue
        
    if user_input.strip().upper() == "END":
        print("\n\033[92mPro:\033[0m Catch you later! Here is your final RFx:")
        agent.view_rfx()
        break
        
    if user_input.strip().upper() == "VIEW":
        agent.view_rfx()
        continue
        
    agent.chat(user_input)

In [ ]:
!pip install python-docx


In [ ]:
# ==============================================================================
# PHASE 6: ADVANCED ANALYST PIPELINE (STANDALONE MODE)
# ==============================================================================
# Run this cell if you have a raw RFQ.pdf and a folder of vendor Proposals 
# downloaded locally. This pipeline parses the RFQ, extracts the proposals, 
# and runs the advanced compliance and scenario modeling engine.
import os
import re
import json
import pandas as pd
from pydantic import BaseModel, Field
from typing import List, Optional
import fitz  # PyMuPDF
from docx import Document
from app import llm
class ExtractedLineItem(BaseModel):
    item_description: str = Field(description="Description of the item quoted. MUST map to an item in the RFQ.")
    location: str = Field(description="Specific delivery location. Split into separate rows for each city if vendor aggregates them.")
    quantity: float = Field(description="Quantity quoted for this specific location.")
    uom: str = Field(description="Unit of measure (e.g., 'Each', 'Box of 10')")
    unit_price: float = Field(description="Unit price quoted")
    currency: str = Field(description="Currency of the quote (e.g., 'INR', 'USD')")
    extraction_confidence: str = Field(description="Rate extraction confidence as 'High', 'Medium', or 'Low' (e.g. Low if reading a blurry photo or handwriting).")
class QuestionnaireAnswer(BaseModel):
    question: str
    answer: str
    confidence_score: float = Field(description="Rate the vendor's answer on a scale of 1-10. Detailed, firm commitments score 10. Vague, evasive, or conditional answers score lower.")
    confidence_reasoning: str = Field(description="A one sentence justification for the confidence score.")
class VendorProposal(BaseModel):
    vendor_name: str
    line_items: List[ExtractedLineItem]
    questionnaire_answers: List[QuestionnaireAnswer]
    lead_time_days: Optional[int] = Field(description="Lead time in days, if mentioned.")
    payment_terms: Optional[str] = Field(description="e.g. Net 30, Net 15")
    freight_included: Optional[bool] = Field(description="True if freight/shipping is included in the price, False if extra.")
    bundle_conditions: Optional[str] = Field(description="Any conditional discounts (e.g., 'Discount applies only if awarded all items').")
    special_notes: Optional[str] = Field(description="Any other footnotes or penalties rejected")
    discount_percentage: float = Field(description="Extract any discount percentage offered on the grand total. 0 if none.", default=0.0)
class RFQSiteQuantity(BaseModel):
    location: str
    quantity: float
class RFQItem(BaseModel):
    generic_requirement: str = Field(description="The core description of the item")
    site_quantities: List[RFQSiteQuantity] = Field(description="Quantities requested per location")
class RFQState(BaseModel):
    scope: str = Field(description="The overarching goal/project scope")
    items: List[RFQItem]
def read_file_to_llm_parts(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext in ['.txt', '.eml']:
        with open(filepath, 'r') as f: return [llm.text_part(f.read())]
    elif ext == '.docx':
        doc = Document(filepath)
        return [llm.text_part("\n".join([p.text for p in doc.paragraphs]))]
    elif ext == '.xlsx':
        df = pd.read_excel(filepath)
        return [llm.text_part(df.to_markdown())]
    elif ext in ['.jpeg', '.jpg', '.png']:
        with open(filepath, 'rb') as f:
            return [llm.image_part(f.read(), mime=f"image/{ext[1:].replace('jpg', 'jpeg')}")]
    elif ext == '.pdf':
        try:
            import pymupdf4llm
            md_text = pymupdf4llm.to_markdown(filepath)
            if len(md_text.strip()) > 100:
                return [llm.text_part(md_text)]
        except Exception:
            pass
            
        parts = []
        doc = fitz.open(filepath)
        for page in doc:
            pix = page.get_pixmap(dpi=150)
            parts.append(llm.image_part(pix.tobytes("png"), mime="image/png"))
        return parts
    return []
# ==============================================================================
# PIPELINE EXECUTION
# ==============================================================================
rfq_path = os.path.expanduser("~/Downloads/RFQ.pdf")
proposals_dir = os.path.expanduser("~/Downloads/Proposals/")
if not os.path.exists(rfq_path) or not os.path.exists(proposals_dir):
    print(f"Waiting for files... Please ensure \n1. RFQ exists at: {rfq_path} \n2. Proposals exist at: {proposals_dir}")
else:
    # STEP 1: Parse the Official RFQ
    print(f"--- STEP 1: Parsing Official RFQ ({rfq_path}) ---")
    rfq_parts = read_file_to_llm_parts(rfq_path)
    instructions = "You are an expert procurement AI. Extract the core requirements, items, and location quantities from this official RFQ document."
    
    rfx_state_parsed, _ = llm.call(
        job="parse_rfq", 
        prompt_version="v2", 
        schema=RFQState, 
        instructions=instructions, 
        content=rfq_parts,
        use_cache=True
    )
    
    rfx_context_str = rfx_state_parsed.model_dump_json(indent=2)
    print("Successfully reverse-engineered RFQ!")
    # Reconstruct the expected locations and line items
    required_locations = set()
    total_rfq_qty = 0
    for item in rfx_state_parsed.items:
        for sq in item.site_quantities:
            required_locations.add(sq.location)
            total_rfq_qty += sq.quantity
    # STEP 2: Extract all Proposals
    print(f"\n--- STEP 2: Extracting Vendor Proposals from {proposals_dir} ---")
    extracted_proposals = []
    
    context_instructions = f"""
    You are an expert procurement AI. Extract the vendor proposal details.
    
    CRITICAL INSTRUCTIONS:
    1. CONTEXT AWARENESS: Below is the official RFQ the buyer sent to the vendors:
    {rfx_context_str}
    
    2. MAP LOCATIONS: You MUST map the vendor's line items to the exact locations in the RFQ. 
       If a vendor explicitly restricts their service to specific locations, you MUST ONLY create rows for those specific locations.
       
    3. QUESTIONNAIRE SCORING: For questionnaire_answers, assign a confidence_score (1-10).
    
    4. CONFIDENCE: Set extraction_confidence to Low if reading from blurry handwriting or unclear phrasing.
    """
    files_to_process = []
    for filename in os.listdir(proposals_dir):
        if filename.startswith('.'): continue
        filepath = os.path.join(proposals_dir, filename)
        parts = read_file_to_llm_parts(filepath)
        if not parts: continue
        files_to_process.append((filename, parts))
    import concurrent.futures
    def extract_single_vendor(args):
        filename, parts = args
        print(f"Extracting {filename}...")
        try:
            parsed, _ = llm.call(
                job="extract_proposal", 
                prompt_version="v6", 
                schema=VendorProposal, 
                instructions=context_instructions, 
                content=parts,
                use_cache=True
            )
            parsed.vendor_name = filename.split('.')[0].replace('_', ' ')
            return parsed
        except Exception as e:
            print(f"Failed to extract {filename}: {e}")
            return None
    with concurrent.futures.ThreadPoolExecutor(max_workers=4) as executor:
        results = executor.map(extract_single_vendor, files_to_process)
        for res in results:
            if res:
                extracted_proposals.append(res)
    # ==============================================================================
    # MATH, AUDITS, AND SCENARIOS ENGINE
    # ==============================================================================
    print(f"\n--- STEP 3: Running Math Engine, Audits & Scenarios ---")
    rows = []
    q_rows = []
    coverage_map = {}
    missing_lines_map = {}
    commercial_terms_map = {}
    for p in extracted_proposals:
        vendor_locations = set()
        vendor_line_combos = set()
        
        for item in p.line_items:
            vendor_locations.add(item.location)
            vendor_line_combos.add((item.item_description.lower(), item.location.lower()))
            
            conversion_rate = 84.0 if item.currency.upper() == 'USD' else 1.0
            norm_price = item.unit_price * conversion_rate
            
            uom_str = item.uom.lower()
            uom_multiplier = 1.0
            if "box of" in uom_str:
                match = re.search(r'box of (\d+)', uom_str)
                if match: uom_multiplier = float(match.group(1))
            
            norm_qty = item.quantity * uom_multiplier
            per_piece_price = norm_price / uom_multiplier
            extended_cost = norm_qty * per_piece_price
            
            rows.append({
                "Vendor": p.vendor_name,
                "Item": item.item_description,
                "Location": item.location,
                "Raw Qty": item.quantity,
                "Raw UOM": item.uom,
                "Raw Price": item.unit_price,
                "Currency": item.currency,
                "Norm Qty (Pieces)": norm_qty,
                "Norm Unit Price (INR)": per_piece_price,
                "Extended Cost (INR)": extended_cost,
                "Extraction Confidence": item.extraction_confidence
            })
            
        for qa in p.questionnaire_answers:
            q_rows.append({
                "Vendor": p.vendor_name,
                "Question": qa.question,
                "Answer": qa.answer,
                "Confidence Score": qa.confidence_score,
                "Confidence Reasoning": qa.confidence_reasoning
            })
            
        # Audits
        delivers_to_all = True if required_locations and required_locations.issubset(vendor_locations) else False
        coverage_map[p.vendor_name] = delivers_to_all
        
        total_vendor_qty = sum(item.quantity * (float(re.search(r'box of (\d+)', item.uom.lower()).group(1)) if "box of" in item.uom.lower() and re.search(r'box of (\d+)', item.uom.lower()) else 1.0) for item in p.line_items)
        
        shortfall = total_rfq_qty - total_vendor_qty
        missing_lines_map[p.vendor_name] = max(0, shortfall)
        commercial_terms_map[p.vendor_name] = {
            "Lead Time (Days)": p.lead_time_days,
            "Payment Terms": p.payment_terms,
            "Freight Included": p.freight_included,
            "Bundle Conditions": p.bundle_conditions,
            "Discount %": p.discount_percentage
        }
    df_proposals = pd.DataFrame(rows)
    df_questionnaire = pd.DataFrame(q_rows)
    # 1. Single Source Scenario
    vendor_totals = df_proposals.groupby('Vendor')['Extended Cost (INR)'].sum().reset_index()
    comm_df = pd.DataFrame.from_dict(commercial_terms_map, orient='index').reset_index().rename(columns={'index': 'Vendor'})
    vendor_totals = pd.merge(vendor_totals, comm_df, on='Vendor')
    vendor_totals['Final Landed Cost (INR)'] = vendor_totals['Extended Cost (INR)'] * (1 - vendor_totals['Discount %'] / 100)
    coverage_df = pd.DataFrame(list(coverage_map.items()), columns=['Vendor', 'Delivers to All Locations'])
    missing_df = pd.DataFrame(list(missing_lines_map.items()), columns=['Vendor', 'Quantity Shortfall'])
    vendor_totals = pd.merge(vendor_totals, coverage_df, on='Vendor')
    vendor_totals = pd.merge(vendor_totals, missing_df, on='Vendor')
    if not df_questionnaire.empty:
        avg_conf = df_questionnaire.groupby('Vendor')['Confidence Score'].mean().round(1).reset_index()
        avg_conf.rename(columns={'Confidence Score': 'Avg QA Confidence / 10'}, inplace=True)
        vendor_totals = pd.merge(vendor_totals, avg_conf, on='Vendor', how='left')
        
    # Knock-outs
    vendor_totals['Compliance Status'] = 'PASS'
    vendor_totals.loc[vendor_totals['Quantity Shortfall'] > 0, 'Compliance Status'] = 'FAIL (Quantity Shortfall)'
    if 'Avg QA Confidence / 10' in vendor_totals.columns:
        vendor_totals.loc[vendor_totals['Avg QA Confidence / 10'] < 6.0, 'Compliance Status'] = 'FAIL (Low QA Score)'
    vendor_totals = vendor_totals.sort_values(['Compliance Status', 'Final Landed Cost (INR)'])
    print("\n" + "="*80)
    print("PHASE 6: SCENARIO A - SINGLE SOURCE (COMPLIANT VENDORS ONLY)")
    print("="*80)
    display(vendor_totals[vendor_totals['Compliance Status'] == 'PASS'])
    print("\n" + "="*80)
    print("PHASE 6: FAILED COMPLIANCE KNOCK-OUTS")
    print("="*80)
    display(vendor_totals[vendor_totals['Compliance Status'] != 'PASS'][['Vendor', 'Compliance Status', 'Quantity Shortfall', 'Avg QA Confidence / 10', 'Bundle Conditions']])
    # 2. Cherry-Pick Scenario
    cherry_pick_df = None
    if not df_proposals.empty:
        passing_vendors = vendor_totals[(vendor_totals['Compliance Status'] == 'PASS') & (vendor_totals['Bundle Conditions'].isna() | (vendor_totals['Bundle Conditions'] == ''))]['Vendor'].tolist()
        if not passing_vendors:
            passing_vendors = vendor_totals['Vendor'].tolist()
            
        eligible_proposals = df_proposals[df_proposals['Vendor'].isin(passing_vendors)]
        
        idx = eligible_proposals.groupby(['Item', 'Location'])['Norm Unit Price (INR)'].idxmin()
        cherry_pick_df = eligible_proposals.loc[idx].copy()
        cherry_pick_df['Cherry Pick Cost'] = cherry_pick_df['Norm Qty (Pieces)'] * cherry_pick_df['Norm Unit Price (INR)']
        total_cherry_pick_cost = cherry_pick_df['Cherry Pick Cost'].sum()
        
        best_single_source_cost = vendor_totals[vendor_totals['Compliance Status'] == 'PASS']['Final Landed Cost (INR)'].min()
        if pd.isna(best_single_source_cost):
             best_single_source_cost = vendor_totals['Final Landed Cost (INR)'].min()
             
        cherry_pick_savings = best_single_source_cost - total_cherry_pick_cost
        print("\n" + "="*80)
        print(f"PHASE 6: SCENARIO B - CHERRY PICK (TOTAL: {total_cherry_pick_cost:,.2f} INR)")
        print(f"POTENTIAL SAVINGS VS BEST SINGLE-SOURCE: {cherry_pick_savings:,.2f} INR")
        print("="*80)
        display(cherry_pick_df[['Item', 'Location', 'Vendor', 'Norm Unit Price (INR)', 'Cherry Pick Cost', 'Extraction Confidence']])


In [ ]:
df_proposals.to_clipboard()

In [ ]:
df_proposals['Notes'].iloc[10]

In [ ]:
# ==============================================================================
# PHASE 4: ANALYST PIPELINE (STANDALONE MODE)
# ==============================================================================
# Run this cell if you have a raw RFQ.pdf and a folder of vendor Proposals 
# downloaded locally (e.g. from an email or a portal) and you want to bypass 
# the Chatbot completely to generate a comparison grid!
import os
import re
import json
import pandas as pd
from pydantic import BaseModel, Field
from typing import List, Optional
import fitz  # PyMuPDF
from docx import Document
from app import llm

# --- 1. Schemas for RFQ Reverse-Engineering ---
class RFQSiteQuantity(BaseModel):
    location: str
    quantity: float

class RFQItem(BaseModel):
    generic_requirement: str = Field(description="The core description of the item")
    site_quantities: List[RFQSiteQuantity] = Field(description="Quantities requested per location")

class RFQState(BaseModel):
    scope: str = Field(description="The overarching goal/project scope")
    items: List[RFQItem]

# --- 2. Schemas for Proposal Extraction ---
class ExtractedLineItem(BaseModel):
    item_description: str = Field(description="Description of the item quoted. MUST map to an item in the RFQ.")
    location: str = Field(description="Specific delivery location. Split into separate rows for each city if vendor aggregates them.")
    quantity: float = Field(description="Quantity quoted for this specific location.")
    uom: str = Field(description="Unit of measure (e.g., 'Each', 'Box of 10')")
    unit_price: float = Field(description="Unit price quoted")
    currency: str = Field(description="Currency of the quote (e.g., 'INR', 'USD')")

class QuestionnaireAnswer(BaseModel):
    question: str
    answer: str

class VendorProposal(BaseModel):
    vendor_name: str
    line_items: List[ExtractedLineItem]
    questionnaire_answers: List[QuestionnaireAnswer]
    special_notes: Optional[str] = Field(description="Any special discounts, penalties rejected, or footnotes")
    discount_percentage: float = Field(description="Extract any discount percentage offered on the grand total. 0 if none.", default=0.0)

def read_file_to_llm_parts(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext in ['.txt', '.eml']:
        with open(filepath, 'r') as f: return [llm.text_part(f.read())]
    elif ext == '.docx':
        doc = Document(filepath)
        return [llm.text_part("\n".join([p.text for p in doc.paragraphs]))]
    elif ext == '.xlsx':
        df = pd.read_excel(filepath)
        return [llm.text_part(df.to_markdown())]
    elif ext in ['.jpeg', '.jpg', '.png']:
        with open(filepath, 'rb') as f:
            return [llm.image_part(f.read(), mime=f"image/{ext[1:].replace('jpg', 'jpeg')}")]
    elif ext == '.pdf':
        parts = []
        doc = fitz.open(filepath)
        for page in doc:
            pix = page.get_pixmap(dpi=150)
            parts.append(llm.image_part(pix.tobytes("png"), mime="image/png"))
        return parts
    return []

# --- 3. Pipeline Execution ---
rfq_path = os.path.expanduser("~/Downloads/RFQ.pdf")
proposals_dir = os.path.expanduser("~/Downloads/Proposals/")

if not os.path.exists(rfq_path) or not os.path.exists(proposals_dir):
    print(f"Waiting for files... Please ensure \n1. RFQ exists at: {rfq_path} \n2. Proposals exist at: {proposals_dir}")
else:
    # STEP 1: Parse the Official RFQ
    print(f"--- STEP 1: Parsing Official RFQ ({rfq_path}) ---")
    rfq_parts = read_file_to_llm_parts(rfq_path)
    instructions = "You are an expert procurement AI. Extract the core requirements, items, and location quantities from this official RFQ document."
    
    rfx_state_parsed, _ = llm.call(
        job="parse_rfq", 
        prompt_version="v1", 
        schema=RFQState, 
        instructions=instructions, 
        content=rfq_parts,
        use_cache=True
    )
    
    rfx_context_str = rfx_state_parsed.model_dump_json(indent=2)
    print("Successfully reverse-engineered RFQ!")

    # STEP 2: Extract all Proposals
    print(f"\n--- STEP 2: Extracting Vendor Proposals from {proposals_dir} ---")
    extracted_proposals = []
    
    context_instructions = f"""
    You are an expert procurement AI. Extract the vendor proposal details.
    
    CRITICAL INSTRUCTIONS:
    1. CONTEXT AWARENESS: Below is the official RFQ the buyer sent to the vendors:
    {rfx_context_str}
    
    2. MAP LOCATIONS: You MUST map the vendor's line items to the exact locations in the RFQ. 
       If a vendor says "We supply all locations", you MUST create a separate line item row for EVERY location listed in the RFQ for that item, assigning the correct quantity for that specific location. Do NOT write "All Locations" or "Central Warehouse" if the RFQ requested specific cities.
       If a vendor explicitly restricts their service to specific locations (e.g., "Only Mumbai and Pune") in their text, you MUST ONLY create rows for those specific locations.
       
    3. EXACT MAPPING: Ensure the quantities and descriptions align with the RFQ's expectations.
    """

    for filename in os.listdir(proposals_dir):
        if filename.startswith('.'): continue
        filepath = os.path.join(proposals_dir, filename)
        parts = read_file_to_llm_parts(filepath)
        if not parts: continue
            
        print(f"Extracting {filename}...")
        try:
            parsed, _ = llm.call(
                job="extract_proposal", 
                prompt_version="v4", 
                schema=VendorProposal, 
                instructions=context_instructions, 
                content=parts,
                use_cache=True
            )
            parsed.vendor_name = filename.split('.')[0].replace('_', ' ')
            extracted_proposals.append(parsed)
        except Exception as e:
            print(f"Failed to extract {filename}: {e}")

    # STEP 3: The Math Engine
    print(f"\n--- STEP 3: Running Math Engine ---")
    rows = []
    for p in extracted_proposals:
        for item in p.line_items:
            conversion_rate = 84.0 if item.currency.upper() == 'USD' else 1.0
            norm_price = item.unit_price * conversion_rate
            
            uom_str = item.uom.lower()
            uom_multiplier = 1.0
            if "box of" in uom_str:
                match = re.search(r'box of (\d+)', uom_str)
                if match: uom_multiplier = float(match.group(1))
            
            norm_qty = item.quantity * uom_multiplier
            per_piece_price = norm_price / uom_multiplier
            extended_cost = norm_qty * per_piece_price
            
            rows.append({
                "Vendor": p.vendor_name,
                "Item": item.item_description,
                "Location": item.location,
                "Raw Qty": item.quantity,
                "Raw UOM": item.uom,
                "Raw Price": item.unit_price,
                "Currency": item.currency,
                "Norm Qty (Pieces)": norm_qty,
                "Norm Unit Price (INR)": per_piece_price,
                "Extended Cost (INR)": extended_cost,
                "Discount %": p.discount_percentage,
                "Notes": p.special_notes
            })

    df_analyst = pd.DataFrame(rows)
    vendor_totals = df_analyst.groupby('Vendor')['Extended Cost (INR)'].sum().reset_index()
    discounts = df_analyst.groupby('Vendor')['Discount %'].max().reset_index()
    vendor_totals = pd.merge(vendor_totals, discounts, on='Vendor')
    vendor_totals['Final Landed Cost (INR)'] = vendor_totals['Extended Cost (INR)'] * (1 - vendor_totals['Discount %'] / 100)
    vendor_totals = vendor_totals.sort_values('Final Landed Cost (INR)')

    print("\n" + "="*80)
    print("PHASE 4 COMPLETE: FINAL VENDOR RANKINGS")
    print("="*80)
    display(vendor_totals)

    print("\n" + "="*80)
    print("NORMALIZED LINE ITEMS GRID")
    print("="*80)
    display(df_analyst)

